# LangGraph 03 · 流式输出与中断（Streaming & Human-in-the-loop）

上一课 `02_记忆_短期与长期.ipynb` 解决了「图怎么记住上一轮」。这一课解决另外两件生产级大事：

1. **流式输出**：大模型逐 token 生成，别让用户盯着空屏等全量结果，而是边跑边吐；
2. **中断（Human-in-the-loop）**：敏感操作（删库、转账、发邮件）不能由 Agent 自作主张，
   必须停在半路，把人请进来点「同意 / 拒绝」再继续。

本节的四个核心概念：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 流式 stream | 把图执行中间产物实时吐出来 | `graph.stream(stream_mode=...)` |
| 动态中断 interrupt() | 节点执行到一半暂停，把问题抛给人类 | `interrupt(payload)` + `Command(resume=值)` |
| 静态断点 interrupt_before | 编译时指定「在某个节点之前停」 | `compile(interrupt_before=["step"])` |
| 检查点 checkpointer | 保存「中断现场」，恢复时靠它找到断点 | `MemorySaver()` / `InMemorySaver()` |

> **本 notebook 由 `Agent/01_langgraph/` 下 7 个脚本合并而成**：
> `05_流式输出.py` + `05_流式输出_jxsd.py`（流式）、
> `06_中断_人工审核.py` + `06_中断_人工审核_jxsd.py`（中断·脚本版）、
> `07_中断_接口版.py` + `07_中断_接口版_jxsd.py`（中断·接口版）、
> `12_记忆_持久化与中断进阶_官方补充.py`（官方文档补充）。

**官方文档**
- 流式输出：<https://docs.langchain.com/oss/python/langgraph/streaming>
- 中断 / 人工审核：<https://docs.langchain.com/oss/python/langgraph/interrupts>
- 持久化 / 检查点：<https://docs.langchain.com/oss/python/langgraph/persistence>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 接口版两节要在 notebook 内自起 FastAPI 服务（末尾自动关掉） |
| 依赖 | `langgraph` / `langchain` / `fastapi` / `uvicorn` / `requests`（venv 已装） |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（流式两节会真实调用大模型） |
| 前置服务 | 无（FastAPI 由 notebook 自己后台拉起，端口 8110 / 8021） |
| 预计耗时 | 约 60～120 秒（流式两节共 6 次真实模型调用，其中「500 字文章」那笔最慢） |

> 本课是 `01_langgraph/` 里**混合档位**的一课：流式两节要模型（🟡），中断脚本版与官方补充
> 完全离线（🟢），接口版要起 FastAPI 服务（🔴）。三档各占一段，正好把上一课到这一课
> 「从纯调度 → 带模型 → 带服务」的进阶线走完。

## 本节地图

先看本课在整章里的位置，以及「中断」这件事在三个脚本里的递进关系。

```mermaid
graph TD
    A["02 记忆<br/>图记得住上一轮"] --> B["03 流式<br/>graph.stream 边跑边吐"]
    B --> C["03 中断·脚本版<br/>interrupt() + Command(resume)"]
    C --> D["03 中断·接口版<br/>FastAPI 把「按回车」换成「点按钮」"]
    D --> E["04 时间旅行<br/>回到历史检查点"]
    C -.-> F["官方补充 12<br/>多中断并行 / 工具内中断 / 检查点粒度"]
```

裸 JupyterLab 不渲染 mermaid，看这张等价表即可：

| 从 | 到 | 讲什么 |
|---|---|---|
| 流式·原版 | 流式·完整版 | 4 种 `stream_mode` 的最小演示 → 三模式逐一遍历对比 |
| 中断·脚本原版 | 中断·脚本完整版 | 动态 `interrupt()` → 静态 `interrupt_before` |
| 中断·接口原版 | 中断·接口完整版 | `/task`+`/review` 常驻服务 → `/start`+`/resume`+`/health` 自测脚手架 |
| 官方补充 12 | — | 记忆裁剪/删除/摘要、Durability 三档、多中断并行、工具内中断 |

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课接口版还要往 `WORKDIR` 下写一个 `interrupt_api/app.py`（后台服务），
> 所以这一格给出的 `WORKDIR` 变量也是硬依赖 —— 见第 5 节。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

In [ ]:
# ===== 前置条件自检（缺了就打印中文提示并让后续相关演示跳过）=====
from config import settings

# 流式两节需要模型配置；接口版需要 fastapi / uvicorn / requests
_MISSING_MODEL = not (settings.api_key and settings.base_url and settings.model_name)
try:
    import fastapi  # noqa: F401
    import uvicorn  # noqa: F401
    import requests  # noqa: F401
    _MISSING_SVC = False
except ImportError as _exc:  # pragma: no cover - 环境缺包时才会走到
    _MISSING_SVC = True
    _MISSING_SVC_NAME = getattr(_exc, "name", str(_exc))

# 接口版后台服务的进程句柄，统一在这里初始化，末尾 taskkill 用
_api_server = None

if _MISSING_MODEL:
    print("[跳过] 未配置大模型（.env 缺 API_KEY / BASE_URL / MODEL_NAME），流式两节无法调用模型。")
if _MISSING_SVC:
    print(f"[跳过] 缺少接口版依赖 {_MISSING_SVC_NAME}（安装：uv add fastapi uvicorn requests）。")
if not _MISSING_MODEL and not _MISSING_SVC:
    print("前置条件自检通过：模型已配置；fastapi / uvicorn / requests 已装。")

## 1. 流式输出：课案原版（4 种 stream_mode 的最小演示）

大模型响应慢，全量等待体验差。LangGraph 支持 4 种流式模式，本课案原版用一个**单节点图**
（`chat` 节点直接 `llm.invoke`）把它们各演示一遍：

| stream_mode | 每次流出什么 | 典型用途 |
|---|---|---|
| `"values"` | 每个节点执行后的**完整状态** | 实时展示完整上下文快照 |
| `"updates"` | 每个节点执行后的**增量更新** | 进度条 /「正在执行 XX 节点」 |
| `"messages"` | LLM 的**单个 token**（打字机） | 打字机效果（体验最好） |
| `"custom"` | 节点内 `get_stream_writer()` 推的内容 | 自定义进度（本节不展开） |

多种模式可同时开启，事件会带 `(模式, 数据)` 元组一起流出。先看模型和这张单节点图：

In [ ]:
# ---------- 1.1 模型 + 单节点图（课案原文） ----------
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, MessagesState, StateGraph

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    streaming=True,  # 开启流式
)


def chat(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
graph = builder.compile()

下面三个函数各演示一种模式。注意 `stream_values` / `stream_updates` 是
`for chunk in graph.stream(...)`（单元素），而 `stream_messages` 因为是**同时开两种模式**，
所以是 `for mode, chunk in ...`（二元组）。

In [ ]:
# ---------- 1.2 values：每步流出完整状态 ----------
def stream_values():
    """values：每步流出完整状态"""
    print("===== stream_mode=values =====")
    for chunk in graph.stream(
        {"messages": [("user", "用一句话介绍上海")]},
        stream_mode="values",
    ):
        print("完整状态最后一条消息：", chunk["messages"][-1].content)

In [ ]:
# ---------- 1.3 updates：每步只流出节点增量 ----------
def stream_updates():
    """updates：每步只流出节点增量"""
    print("===== stream_mode=updates =====")
    for chunk in graph.stream(
        {"messages": [("user", "用一句话介绍北京")]},
        stream_mode="updates",
    ):
        print("节点增量：", chunk)

In [ ]:
# ---------- 1.4 messages：token 级流式，打字机效果 ----------
def stream_messages():
    """messages：token 级流式，打字机效果"""
    print("===== stream_mode=messages =====")
    for mode, chunk in graph.stream(
        {"messages": [("user", "讲一个 50 字左右的笑话")]},
        stream_mode=["messages", "updates"],
    ):
        if mode == "messages":
            # 新版 LangGraph 的 messages 流返回 (消息块, 元数据) 元组
            if isinstance(chunk, tuple):
                chunk = chunk[0]
            # chunk 是 AIMessageChunk，一个 token
            content = getattr(chunk, "content", "")
            if isinstance(content, list):  # 部分模型的 content 是分块列表
                content = "".join(str(c) for c in content)
            if content:
                print(content, end="", flush=True)
        else:
            print()  # 节点结束时换行

依次跑三种模式（课案原版把这三行写在 `if __name__ == "__main__"` 里，这里直接顶格调用）。
**注意 `stream_messages` 会真的逐 token 打印** —— 这就是「打字机」的来源。

In [ ]:
stream_values()
stream_updates()
stream_messages()

### 预期输出

```text
===== stream_mode=values =====
完整状态最后一条消息： 用一句话介绍上海
===== stream_mode=updates =====
===== stream_mode=messages =====
```

⚠️ 上面只列**稳定**的四行：三行 `=====` 标题 + 第一行「用一句话介绍上海」
（那是 `values` 第一个 chunk 的**输入状态**，原样回显）。`updates` 那行打印的
`AIMessage(content=...)` 和 `messages` 那段逐 token 的笑话，都是**模型措辞、每次不同**，
别逐字比对 —— 只看三行的先后顺序，以及「输入 → 模型回复」这个结构。

三个值得停一下看的细节：

1. `values` 打出来的是**完整状态**（`chunk["messages"][-1].content` 取最后一条）；
2. `updates` 打出来的是 `{"节点名": 增量}` 形状的单键字典 —— 只有一个 `chat` 节点；
3. `messages` 打出来的是一个个 token 拼成的文本，`end=""` + `flush=True` 就是打字机的关键。

## 2. 流式输出：完整版（三种 stream_mode 逐一遍历对比）

课案原版把三种模式挤在一个单节点图里；完整版换成一个**三节点直链图**
（`one → two → three`，只有 `three` 节点调模型），这样能讲清一个关键点：
**`messages` 模式是在 LangGraph 层面拦截回调事件**的 —— 节点里用的是 `llm.invoke()`
（同步、非流式），只要发生的是 LangChain 模型调用，LangGraph 就能拿到
`on_chat_model_stream` 事件逐 token 吐出，**业务代码不用改**。

In [ ]:
# ---------- 2.1 模型 + 状态 + 三节点（课案原文） ----------
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# 课案原文写死在 step_three 里的提问词。抽成常量只为「打印出来的提问」和
# 「真正发给模型的提问」保持同一个来源，避免改了一处忘了另一处。
ARTICLE_PROMPT = "请写出一篇500字的文章"


class State(TypedDict):
    """课案原文状态：只有一个 text 字段，节点逐段往后拼接。"""

    text: str


def step_one(state: State) -> dict:
    return {"text": state["text"] + " → 步骤一"}


def step_two(state: State) -> dict:
    return {"text": state["text"] + " → 步骤二"}


def step_three(state: State) -> dict:
    """三个节点里唯一调 LLM 的节点，也是 messages 模式唯一产出 token 的节点。

    ⚠️ 注意这里用的是 `llm.invoke()`（同步、非流式调用）。
    那为什么外层 `stream_mode="messages"` 还能看到一个个 token？
    因为 messages 模式是**在 LangGraph 层面拦截回调事件**的：
    只要节点里发生的是 LangChain 的模型调用，LangGraph 就能拿到
    on_chat_model_stream 事件并逐个吐出 chunk，节点代码本身不用改。
    —— 这是 messages 模式最好用的一点：**不用为了流式去改业务代码**。
    """
    response = llm.invoke(
        [
            {"role": "user", "content": ARTICLE_PROMPT},
        ]
    )
    return {"text": response.content + " → 步骤三"}


builder = StateGraph(State)
builder.add_node("one", step_one)
builder.add_node("two", step_two)
builder.add_node("three", step_three)
builder.add_edge(START, "one")
builder.add_edge("one", "two")
builder.add_edge("two", "three")
builder.add_edge("three", END)
graph = builder.compile()

### 2.1 模式一：messages（课案主推，打字机效果）

`messages` 模式流出的是 `(message_chunk, metadata)` **二元组**。`metadata` 里带
`langgraph_node`（这段 token 是哪个节点产的），图里一旦有多个节点调模型，
就必须靠它把 token 分流 —— 否则几个节点的输出会糊在一起。

In [ ]:
# ---------- 2.2 模式一：messages ----------
def stream_messages() -> None:
    """课案原文实现：只输出 three 节点产生的 LLM 内容。"""
    print("=" * 74)
    print('① stream_mode="messages" —— token 级流式，打字机效果')
    print("=" * 74)
    print(f"  提问：{ARTICLE_PROMPT}")
    print("  输出：", end="", flush=True)

    emitting_nodes: set[str] = set()  # 记录「哪些节点产出过 token」，用于讲清过滤器的价值
    first_metadata: dict | None = None

    for message_chunk, metadata in graph.stream(
        {"text": "开始"},
        stream_mode="messages",
    ):
        # metadata 里带这批 token 的来路信息，第一次拿到时打印出来看看有什么
        if first_metadata is None:
            first_metadata = metadata
            print(f"\n  [调试] 第一个 chunk 的 metadata 键：{sorted(metadata.keys())}\n")

        emitting_nodes.add(metadata.get("langgraph_node"))

        # 只输出 three 节点产生的 LLM 内容
        if metadata.get("langgraph_node") == "three":
            content = message_chunk.content

            if isinstance(content, str) and content:
                print(content, end="", flush=True)

    print("\n")
    print(f"  [观察] 本次产出 token 的节点集合：{emitting_nodes}")
    print("  ↑ 本例只有 three 节点调模型，所以过滤器看着「没起作用」；")
    print("    一旦图里有多个节点调模型（多 Agent 常见），")
    print("    metadata['langgraph_node'] 就是唯一能把 token 分流的手段。")
    print(f"  [观察] metadata 样例：langgraph_node={first_metadata.get('langgraph_node')!r} "
          f"langgraph_step={first_metadata.get('langgraph_step')!r}")
    print()

### 2.2 模式二：updates（每完成一个节点流出该节点的增量）

课案注释里原本是 `print(chunk)`（会把 500 字长文整段刷屏）；完整版只截前 60 字并替换换行。
**流出的数据结构没有改动**，只是显示做了截断。

In [ ]:
# ---------- 2.3 模式二：updates ----------
def stream_updates() -> None:
    """课案注释原文：

    # updates：每完成一个节点就输出该节点的增量
    # for chunk in graph.stream({"text": "开始"}, stream_mode="updates"):
        # print(chunk)  # {'one': {'text': '开始 → 步骤一'}}  →  {'two': {...}}
    """
    print("=" * 74)
    print('② stream_mode="updates" —— 每完成一个节点，流出该节点的增量')
    print("=" * 74)
    print("  输出形如 {\"节点名\": {该节点返回的增量}}，节点没返回值时是 {\"节点名\": None}")
    for chunk in graph.stream({"text": "开始"}, stream_mode="updates"):
        node_name = next(iter(chunk))
        delta = chunk[node_name]
        if isinstance(delta, dict) and "text" in delta:
            # 课案原样是 print(chunk)；这里三个节点的增量都带 text，
            # 而 three 节点那段是 LLM 写的 500 字长文，全打出来会把对比结果刷没，
            # 所以只截前 60 字——流出的数据结构本身没有改动。
            shown = str(delta["text"])[:60].replace("\n", " ")
            print(f"      {node_name:<6} → text 前 60 字：{shown}…")
        else:
            print(f"      {node_name:<6} → {delta}")
    print("\n  用途：做「正在执行 XXX 节点」的进度提示最合适——")
    print("        它只给增量，不需要把整个状态搬来搬去。")
    print()

### 2.3 模式三：values（每完成一个节点流出完整状态）

`values` 的第一个 chunk 是**输入状态**（还没跑任何节点），所以三个节点会给出 **4 个 chunk，
不是 3 个** —— 这是「为什么多了一条」的答案，最容易数错。

In [ ]:
# ---------- 2.4 模式三：values ----------
def stream_values() -> None:
    """课案注释原文：

    # values：每完成一个节点就输出完整的当前状态
    # for chunk in graph.stream({"text": "开始"}, stream_mode="values"):
        # print(chunk["text"])  # 开始 → 开始 → 步骤一 → 开始 → 步骤一 → 步骤二
    """
    print("=" * 74)
    print('③ stream_mode="values" —— 每完成一个节点，流出完整的当前状态')
    print("=" * 74)
    print("  第一个 chunk 是**输入状态**（还没跑任何节点），之后每跑完一个节点再来一个：")
    for i, chunk in enumerate(graph.stream({"text": "开始"}, stream_mode="values")):
        # 课案原样是 print(chunk["text"])；同理这里只截前 70 字防刷屏
        text = str(chunk["text"]).replace("\n", " ")
        print(f"      [{i}] {text[:70]}…")
    print("\n  用途：需要「每步之后的完整快照」时用它（比如把中间状态渲染到前端）；")
    print("        代价是数据量大——状态越大，重复传输越浪费，这也是它和 updates 的核心差别。")
    print()

三种模式依次各跑一遍，对比它们「流出的东西」有什么不同（课案原文写在 `if __name__` 里）。

In [ ]:
print("三种模式依次各跑一遍，对比它们「流出的东西」有什么不同：\n")
stream_messages()  # 课案原文：长文 + 打字机
stream_updates()  # 课案注释：updates
stream_values()  # 课案注释：values
print("=" * 74)
print("小结：messages 给 token（体验最好）、updates 给增量（最省）、values 给全量（最全）。")
print("      三者可以同时开：stream_mode=['messages', 'updates']，")
print("      这时每个事件是 (模式名, 数据) 二元组，按模式名分流即可。")
print("=" * 74)

### 预期输出

```text
三种模式依次各跑一遍，对比它们「流出的东西」有什么不同：
① stream_mode="messages" —— token 级流式，打字机效果
  提问：请写出一篇500字的文章
  输出：
  [观察] 本次产出 token 的节点集合：{'three'}
② stream_mode="updates" —— 每完成一个节点，流出该节点的增量
  输出形如 {"节点名": {该节点返回的增量}}，节点没返回值时是 {"节点名": None}
      one    → text 前 60 字：开始 → 步骤一…
      two    → text 前 60 字：开始 → 步骤一 → 步骤二…
③ stream_mode="values" —— 每完成一个节点，流出完整的当前状态
  第一个 chunk 是**输入状态**（还没跑任何节点），之后每跑完一个节点再来一个：
      [0] 开始…
      [1] 开始 → 步骤一…
      [2] 开始 → 步骤一 → 步骤二…
小结：messages 给 token（体验最好）、updates 给增量（最省）、values 给全量（最全）。
```

⚠️ 上面列的是**稳定**的行；以下这些是**模型措辞 / 元数据，每次不同、别逐字比对**：
① 的 500 字正文、「第一个 chunk 的 metadata 键」整行、`langgraph_step` 的具体数字；
② 的 `three` 行（截的是 500 字正文前 60 字）；③ 的 `[3]` 行（同上）。
结构是稳定的：`messages` 给 token（体验最好）、`updates` 给增量（最省）、
`values` 给全量（最全），且 `values` 有 **4 个 chunk**（多出来的第一个是输入状态）。

## 3. 中断：脚本版课案原版（动态中断 `interrupt()`）

敏感操作不应让 Agent 自作主张。做法：在节点里调用 `interrupt(payload)`，图会在那里暂停，
把「待审核信息」抛给人类；人类通过 `Command` 决定：

- **同意**：`Command(resume=True)` → 从断点继续执行；
- **拒绝**：`Command(resume=错误反馈)` → 节点内拿到反馈走拒绝分支。

两个关键点：**编译时必须传 checkpointer**（中断状态要靠检查点保存）；
`interrupt()` 会在重放时返回 `Command(resume=...)` 传进来的值。

In [ ]:
# ---------- 3.1 状态 + 审核节点 + 执行节点（课案原文） ----------
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class State(TypedDict):
    task: str        # 待执行的任务
    approved: bool   # 是否被人工批准


def human_review(state: State) -> dict:
    """
    人工审核节点。
    interrupt(payload) 暂停图执行，payload 会出现在
    __interrupt__ 事件里，展示给人类审核。
    """
    answer = interrupt(
        {"question": f"要执行任务「{state['task']}」，请确认是否批准？(yes/no)"}
    )
    # 恢复执行后，interrupt() 的返回值就是 Command(resume=...) 传的值
    return {"approved": answer in (True, "yes")}


def execute(state: State) -> dict:
    """执行节点：只有批准了才会真正走到这里"""
    if state["approved"]:
        print(f"✅ 已批准，正在执行：{state['task']}")
    else:
        print(f"❌ 人类已拒绝，任务「{state['task']}」不会执行")
    return {}


builder = StateGraph(State)
builder.add_node("human_review", human_review)
builder.add_node("execute", execute)
builder.add_edge(START, "human_review")
# 条件边：批准 → 执行；拒绝 → 直接结束
builder.add_conditional_edges(
    "human_review",
    lambda s: "execute" if s.get("approved") else END,
    ["execute", END],
)
builder.add_edge("execute", END)

# 中断必须配合 checkpointer
graph = builder.compile(checkpointer=MemorySaver())

先发起一个任务（停在 `interrupt` 处），再拒绝；换一个 `thread_id` 再来一次、这次批准。
注意 `thread_id` 就是「中断现场」的编号，恢复时必须用同一个。

In [ ]:
config = {"configurable": {"thread_id": "review-1"}}

# 第一次 invoke：会停在 interrupt 处
result = graph.invoke({"task": "删除生产数据库里的测试表"}, config)
# 停在断点时，结果里带 __interrupt__ 信息
print("暂停，等待审核：", result["__interrupt__"])

# 人类审核：拒绝（也可以给文字反馈，让 Agent 调整后重试）
result = graph.invoke(Command(resume="no"), config)
print("拒绝后结果：", result)

# ---------- 再来一次，这次批准 ----------
config = {"configurable": {"thread_id": "review-2"}}
graph.invoke({"task": "清理过期缓存"}, config)
result = graph.invoke(Command(resume="yes"), config)
print("批准后结果：", result)

### 预期输出

```text
拒绝后结果： {'task': '删除生产数据库里的测试表', 'approved': False}
✅ 已批准，正在执行：清理过期缓存
批准后结果： {'task': '清理过期缓存', 'approved': True}
```

第一行 `暂停，等待审核：` 里带一个每次运行随机生成的 `id`（UUID），**别逐字比对** ——
它的 `value`（那条「是否批准？」的提问文案）是稳定的。上面三行是稳定输出。

三个细节：

1. 第一次 `invoke` 停在断点，返回值里带 `__interrupt__`，`value` 就是 `interrupt()` 传的 payload；
2. 拒绝（`resume="no"`）后 `approved=False`，`execute` 节点打印「已拒绝」，但**没真正执行**；
3. 批准（`resume="yes"`）后 `approved=True`，`execute` 节点**先**打印「✅ 已批准，正在执行」，
   然后才是 `print("批准后结果：", ...)` —— 所以「✅」那行在「批准后结果」前面。

## 4. 中断：脚本版完整版（静态断点 `interrupt_before`）

课案原句「`interrupt_before` 让图在指定节点前暂停，`Command(resume=...)` 恢复执行」里，
其实藏着**两种中断机制**，别混：

| 对比项 | 静态断点 `interrupt_before` | 动态中断 `interrupt()` |
|---|---|---|
| 断在哪 | 编译时写死，节点执行**之前** | 运行时决定，执行到那一行才停 |
| 带数据吗 | 不带（只是「停一下」） | 能把「待审核的问题」抛给人类 |
| 恢复方式 | `invoke(None, config)` | `invoke(Command(resume=值), config)` |
| 典型场景 | 固定的人工审核关卡 | 按内容决定要不要问人 |

**为什么中断必须配 checkpointer？** 「暂停」要实现成「把现场存下来再退出执行」，
没有 checkpointer 就没有地方存这个现场，图停下就真丢了。

**怎么知道停在哪？** `snapshot = graph.get_state(config)` 看 `snapshot.next`：
`('step',)` 表示卡在 step 前，`()` 表示已到 END。

In [ ]:
# ---------- 4.1 状态 + 节点 + 静态断点图（课案原文） ----------
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    text: str


def step(state: State) -> dict:
    """被中断挡在门外的那个节点。

    打印「step节点开始执行」是课案刻意留的观察点：
    第一次 invoke 时**看不到**它，恢复之后才会出现——一眼就能看出中断生效了。
    """
    print("step节点开始执行")

    return {"text": state["text"] + " → 已执行"}


builder = StateGraph(State)

builder.add_node("step", step)
builder.add_edge(START, "step")
builder.add_edge("step", END)

# 在 step 节点执行之前设置静态断点
# 注意：断点写在 compile() 里，属于「图的结构特性」，编译后不可更改；
#       运行时想动态决定要不要停，就得用 interrupt()（见第 5 节接口版）。
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["step"],
)

# config 里的 thread_id 就是「这次中断现场的编号」。
# 恢复时必须用**同一个 config**，否则 LangGraph 找不到刚才存下的现场。
config = {
    "configurable": {
        "thread_id": "1"
    }
}

课案原样是 `input("...按回车继续...")` 直接等人。完整版包了一层 `wait_for_enter()`：
在 CI / 重定向输入 / IDE 运行窗口等**非交互环境**里 `stdin` 不是终端，自动继续、不会卡死。
（notebook 无头执行正是这种非交互环境，所以这一层兜底在这里是**必须的**。）

In [ ]:
# ---------- 4.2 非交互环境兜底 ----------
def wait_for_enter() -> None:
    """课案原文：input("\\n图已经暂停，按回车继续执行step节点……")

    加两层防护是工程上的必要处理：
      1. sys.stdin.isatty()：在 CI / 重定向输入 / 被其他程序调用时，
         stdin 不是终端，按回车无从谈起，直接自动继续；
      2. except EOFError：有些「伪终端」（IDE 运行窗口、任务编排器）里
         isatty() 会返回 True，但真正读的时候仍然立刻 EOF。
         单靠 isatty() 判断会漏掉这种情况，必须把读取动作本身也兜住。

    课案的演示效果（暂停 + 打印提示）完整保留，只是不会再把程序打断。
    """
    prompt = "\n图已经暂停，按回车继续执行step节点……"
    if not sys.stdin.isatty():
        print(prompt)
        print("（检测到当前是非交互环境 stdin 不是终端，自动继续，避免卡死）")
        return
    try:
        input(prompt)
    except EOFError:
        print("\n（stdin 已到结尾，读不到回车，自动继续，避免卡死）")

主流程五步：① 第一次 invoke 卡在 step 前 → ② 看现场 `next=('step',)` →
③ 恢复执行 → ④ 再看 `next=()` → ⑤ 换 thread_id 再来一次（证明断点对每个 thread_id 都生效）。

In [ ]:
print("=" * 74)
print("① 第一次调用图：会被静态断点挡在 step 节点之前")
print("=" * 74)
print("第一次调用图")

first_result = graph.invoke(
    {"text": "hello"},
    config=config,
)

print("  第一次返回结果：", first_result)
# 预期输出：{'text': 'hello'} —— 输入原样返回，step 节点没跑
print("  ↑ 注意 text 还是 'hello'，没有 ' → 已执行'，说明 step 确实没执行。")
print()

# ---------- 查看暂停位置 ----------
print("=" * 74)
print("② 查看暂停位置：get_state(config) 看「现场」")
print("=" * 74)
snapshot = graph.get_state(config)

print("  当前状态：", snapshot.values)
print("  等待执行的节点：", snapshot.next)
# 预期输出：('step',) —— 下一个待执行节点是 step
print("  ↑ next=('step',) 就是「卡在 step 前面」的书面证据；")
print("    如果已经跑完，next 会是空元组 ()。")
print()

# ---------- 手动等待，便于观察中断 ----------
wait_for_enter()

# ---------- 恢复执行 ----------
print("=" * 74)
print("③ 恢复执行：静态中断用 None 恢复（不是 Command！）")
print("=" * 74)
# 「传 None」的语义是：不提供新输入，从上次保存的现场**接着往下跑**。
# 传 Command(resume=...) 是给**动态中断** interrupt() 传返回值的，用在静态断点上不对。
second_result = graph.invoke(
    None,
    config=config,
)

print("\n  恢复后的结果：", second_result)
# 预期输出：{'text': 'hello → 已执行'}，且上一行会出现「step节点开始执行」
print()

# ---------- 再次查看图状态 ----------
print("=" * 74)
print("④ 再次查看图状态：已到 END，next 变成空元组")
print("=" * 74)
snapshot = graph.get_state(config)

print("  最终状态：", snapshot.values)
print("  后续节点：", snapshot.next)
# 预期输出：next=() —— 没有待执行节点，图已到达 END
print()

# ---------- 补充：同一个断点可以反复用 ----------
print("=" * 74)
print("⑤ 补充：换一个 thread_id，同一个断点又是一次全新审核")
print("=" * 74)
config2 = {"configurable": {"thread_id": "2"}}
r = graph.invoke({"text": "第二次 hello"}, config=config2)
s = graph.get_state(config2)
print(f"  新会话第一次 invoke 返回：{r}")
print(f"  它的 next：{s.next}   ← 同样卡在 step 前面")
print("  ↑ 断点是图的结构属性，对每个 thread_id 都生效；")
print("    这也说明了为什么 thread_id 是「中断现场」的定位键。")
print()
print("=" * 74)
print("小结：静态断点 = 编译时写死的人工关卡，用 invoke(None, config) 放行。")
print("      实际项目里这个「按回车」的动作会变成一个 HTTP 接口 → 见第 5 节接口版。")
print("=" * 74)

### 预期输出

```text
① 第一次调用图：会被静态断点挡在 step 节点之前
第一次调用图
  第一次返回结果： {'text': 'hello'}
  ↑ 注意 text 还是 'hello'，没有 ' → 已执行'，说明 step 确实没执行。

② 查看暂停位置：get_state(config) 看「现场」
  当前状态： {'text': 'hello'}
  等待执行的节点： ('step',)

图已经暂停，按回车继续执行step节点……
（检测到当前是非交互环境 stdin 不是终端，自动继续，避免卡死）

③ 恢复执行：静态中断用 None 恢复（不是 Command！）
step节点开始执行
  恢复后的结果： {'text': 'hello → 已执行'}

④ 再次查看图状态：已到 END，next 变成空元组
  最终状态： {'text': 'hello → 已执行'}
  后续节点： ()

⑤ 补充：换一个 thread_id，同一个断点又是一次全新审核
  新会话第一次 invoke 返回：{'text': '第二次 hello'}
  它的 next：('step',)   ← 同样卡在 step 前面
```

最关键的观察：第一次 invoke **没有**「step节点开始执行」，恢复之后**才出现** ——
这就是中断生效的直接证据。而 `next` 从 `('step',)` 变成 `()`，说明图确实跑到了 END。

## 5. 中断：接口版课案原版（FastAPI，后台起服务）

真实系统里「按回车继续」这个动作不是人坐在终端前敲的，而是**审核人在前端点了个按钮**，
前端再调一个 HTTP 接口。于是「中断现场」必须跨请求保存在服务端 —— 这正是 checkpointer 的用武之地：

    用户 A 调 POST /task    → 图跑到断点挂起，接口返回 waiting_review
    （等待人工审核……可能是几秒，也可能是几小时）
    审核人调 POST /review  → 图从断点继续跑完，接口返回 done

⚠️ **本课案原版是 `uvicorn.run(...)` 阻塞常驻**，直接跑会把 notebook 内核永久卡死。
所以这里按规范改成：**把 app 落成一个独立 `app.py`，用 `subprocess.Popen` 后台拉起，
端口从 8000 改成 8110**（本机 8100 已被 RAG 的 graph_rag 服务占用），轮询等就绪，末尾 `taskkill` 关掉。
下面先把课案原版的 FastAPI 应用完整定义出来：

In [ ]:
# ---------- 5.1 FastAPI 应用（课案原文，接口版 /task + /review） ----------
from typing import TypedDict

from fastapi import FastAPI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

app = FastAPI(title="人工审核示例")


class State(TypedDict):
    task: str
    approved: bool


def human_review(state: State) -> dict:
    answer = interrupt({"question": f"是否批准任务：{state['task']}？"})
    return {"approved": answer in (True, "yes", "True")}


def execute(state: State) -> dict:
    print(f"执行任务：{state['task']}")
    return {}


builder = StateGraph(State)
builder.add_node("human_review", human_review)
builder.add_node("execute", execute)
builder.add_edge(START, "human_review")
builder.add_conditional_edges(
    "human_review",
    lambda s: "execute" if s.get("approved") else END,
    ["execute", END],
)
builder.add_edge("execute", END)

graph = builder.compile(checkpointer=MemorySaver())


class TaskRequest(BaseModel):
    thread_id: str
    task: str


class ReviewRequest(BaseModel):
    thread_id: str
    approved: bool


@app.post("/task")
def create_task(req: TaskRequest):
    """发起任务；如果触发审核，返回中断信息给前端展示审核弹窗"""
    config = {"configurable": {"thread_id": req.thread_id}}
    result = graph.invoke({"task": req.task}, config)
    if "__interrupt__" in result:
        return {"status": "waiting_review", "interrupt": result["__interrupt__"]}
    return {"status": "done"}


@app.post("/review")
def review(req: ReviewRequest):
    """提交审核结果，恢复被中断的图"""
    config = {"configurable": {"thread_id": req.thread_id}}
    result = graph.invoke(Command(resume=req.approved), config)
    return {"status": "done", "result": {k: v for k, v in result.items() if k != "__interrupt__"}}

上面定义的 `app` 只能在同一进程内跑。要后台起服务，得把它**序列化成一个独立 `app.py`**
（内容与上面一字不差，只是把端口写成 8110），再用 `subprocess.Popen` 后台拉起，
并轮询端口就绪（不写死 sleep）。

In [ ]:
# ---------- 5.2 写 app.py + 后台起服务（端口 8110）----------
import socket
import subprocess
import sys
import time
import os

API_DIR = WORKDIR / "interrupt_api"   # 本 notebook 专属子目录，避免和其它 notebook 撞文件
API_DIR.mkdir(exist_ok=True)

APP_FILE = API_DIR / "app.py"
APP_FILE.write_text('''\
# -*- coding: utf-8 -*-
"""人工审核接口服务（由 notebook 现场生成，端口 8110）。"""
from typing import TypedDict

from fastapi import FastAPI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

app = FastAPI(title="人工审核示例")


class State(TypedDict):
    task: str
    approved: bool


def human_review(state: State) -> dict:
    answer = interrupt({"question": f"是否批准任务：{state['task']}？"})
    return {"approved": answer in (True, "yes", "True")}


def execute(state: State) -> dict:
    print(f"执行任务：{state['task']}")
    return {}


builder = StateGraph(State)
builder.add_node("human_review", human_review)
builder.add_node("execute", execute)
builder.add_edge(START, "human_review")
builder.add_conditional_edges(
    "human_review",
    lambda s: "execute" if s.get("approved") else END,
    ["execute", END],
)
builder.add_edge("execute", END)

graph = builder.compile(checkpointer=MemorySaver())


class TaskRequest(BaseModel):
    thread_id: str
    task: str


class ReviewRequest(BaseModel):
    thread_id: str
    approved: bool


@app.post("/task")
def create_task(req: TaskRequest):
    config = {"configurable": {"thread_id": req.thread_id}}
    result = graph.invoke({"task": req.task}, config)
    if "__interrupt__" in result:
        return {"status": "waiting_review", "interrupt": result["__interrupt__"]}
    return {"status": "done"}


@app.post("/review")
def review(req: ReviewRequest):
    config = {"configurable": {"thread_id": req.thread_id}}
    result = graph.invoke(Command(resume=req.approved), config)
    return {"status": "done", "result": {k: v for k, v in result.items() if k != "__interrupt__"}}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8110)
''', encoding="utf-8")

# 后台拉起：PYTHONUTF8=1（中文 Windows 防 GBK 报错）+ NO_PROXY（本机 Clash 拦回环）
env = {**os.environ, "PYTHONUTF8": "1", "NO_PROXY": "127.0.0.1,localhost"}
_api_server = subprocess.Popen(
    [sys.executable, str(APP_FILE)],
    cwd=str(ROOT), env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)


def _port_open(port: int, timeout: float = 0.5) -> bool:
    """单次探测：端口是否已经能连上（不等待）。"""
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=timeout):
            return True
    except OSError:
        return False


# 轮询等端口就绪（最多 30 秒，不把 sleep 写死很久）
_ready = False
for _ in range(60):
    if _port_open(8110):
        _ready = True
        break
    time.sleep(0.5)
print("8110 服务就绪：", _ready)

服务起来了，用 `requests` 当客户端打一遍课案的两个接口：先 `/task` 发起任务（应停在审核），
再 `/review` 批准 / 拒绝。第二个任务演示拒绝分支。

In [ ]:
# ---------- 5.3 打接口：/task 触发审核 + /review 批准/拒绝 ----------
import requests

BASE = "http://127.0.0.1:8110"

if _ready:
    r = requests.post(f"{BASE}/task", json={"thread_id": "api-1", "task": "删除生产库的测试表"}, timeout=15)
    print("POST /task（任务1，应停在审核）→", r.status_code, r.json())

    r = requests.post(f"{BASE}/review", json={"thread_id": "api-1", "approved": True}, timeout=15)
    print("POST /review（任务1，批准）→", r.status_code, r.json())

    r = requests.post(f"{BASE}/task", json={"thread_id": "api-2", "task": "清理过期缓存"}, timeout=15)
    print("POST /task（任务2，应停在审核）→", r.status_code, r.json())

    r = requests.post(f"{BASE}/review", json={"thread_id": "api-2", "approved": False}, timeout=15)
    print("POST /review（任务2，拒绝）→", r.status_code, r.json())
else:
    print("[跳过] 8110 服务未就绪，跳过接口调用演示。")

### 预期输出

```text
8110 服务就绪： True
POST /review（任务1，批准）→ 200 {'status': 'done', 'result': {'task': '删除生产库的测试表', 'approved': True}}
POST /review（任务2，拒绝）→ 200 {'status': 'done', 'result': {'task': '清理过期缓存', 'approved': False}}
```

⚠️ 两次 `/task` 都返回 `status='waiting_review'` + `interrupt`，其中 `interrupt[0].id` 是
每次运行随机生成的 UUID（**别逐字比对**），`interrupt[0].value.question` 是稳定文案。
`/review` 批准后 `result.approved=True`（`execute` 节点跑过）、拒绝后 `approved=False`（没跑）。
两个 `thread_id` 各自一份中断现场，互不干扰 —— 这就是 checkpointer 在跨请求场景下的作用。

## 6. 中断：接口版完整版（自测脚手架，`/start` + `/resume` + `/health`）

课案原版的接口版是「起服务 → 让你用浏览器/Postman 手动点」。完整版在 `__main__` 里
**真的把 uvicorn 起起来、自动打一遍接口、再自己关掉**：既能当教学演示，也能当冒烟测试。

它用的是一个更精细的实现：`interrupt_before` 静态断点 + `/start/{thread_id}`、`/resume/{thread_id}`、
`/health` 三个接口；服务塞在**子线程**里跑（`uvicorn.Server(config).run()`），主线程用 `requests`
当客户端，测完 `server.should_exit = True` 收工。端口用 **8021**（避开第 5 节的 8110）。

In [ ]:
# ---------- 6.1 状态 + 静态断点图（与第 4 节一致，只是把「按回车」换成交互接口） ----------
import time
import threading

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

PORT = 8021  # 刻意避开 8000/8080，防止和本机已有服务撞端口


class State(TypedDict):
    text: str


def step(state: State) -> dict:
    """被人工审核挡住的那个节点。"""
    print("      [服务端] step节点开始执行")
    return {"text": state["text"] + " → 已执行"}


builder = StateGraph(State)
builder.add_node("step", step)
builder.add_edge(START, "step")
builder.add_edge("step", END)

# ⚠️ 接口版**必须**用能跨请求、跨进程的 checkpointer 吗？
#    用 MemorySaver 时：中断现场存在**当前服务进程**的内存里。
#    只要服务不重启，多次 HTTP 请求之间就能接上（这已经能满足「审核人稍后点按钮」）；
#    但如果服务重启（发布、崩溃、多副本负载均衡），现场就丢了。
#    → 生产环境把这里换成 PostgresSaver(settings.pg_uri)，恢复逻辑一行都不用改。
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["step"],
)

In [ ]:
# ---------- 6.2 FastAPI 应用 + 两个接口 + 健康检查（课案原文结构） ----------
from fastapi import FastAPI

app = FastAPI(title="LangGraph 中断接口版")


def get_config(thread_id: str):
    """把 thread_id 包装成 LangGraph 需要的 config 结构。

    抽成函数是为了在 /start 和 /resume 里**用同一个来源**生成 config——
    两边只要有一处写错（比如漏了 configurable 这层），
    就会出现「恢复时找不到现场」这种极难排查的问题。
    """
    return {
        "configurable": {
            "thread_id": thread_id
        }
    }


@app.post("/start/{thread_id}")
def start_graph(thread_id: str):
    """发起任务：图会一路跑到断点前停下。

    返回值里带上 next，前端可以据此知道「卡在哪个节点」，
    如果 next 为空就说明这个任务压根没触发中断（已经跑完了）。
    """
    config = get_config(thread_id)

    result = graph.invoke(
        {"text": "hello"},
        config=config,
    )

    snapshot = graph.get_state(config)

    return {
        "state": result,
        "next": snapshot.next,  # 待执行节点（元组 → JSON 数组）
        "status": "waiting" if snapshot.next else "completed",
    }


@app.post("/resume/{thread_id}")
def resume_graph(thread_id: str):
    """人工审核通过：从断点继续执行。

    ⚠️ 课案原文直接 invoke(None, config)。这里加了一道前置校验：
       如果这个 thread_id 没有处于中断状态（没调过 /start、或者已经恢复过了），
       invoke(None) 会拿一个空状态继续跑，行为不符合预期且难排查。
       HTTP 接口是对外边界，参数校验不能省——所以先查 snapshot.next 再放行。
    """
    config = get_config(thread_id)

    snapshot = graph.get_state(config)
    if not snapshot.next:
        return {
            "state": snapshot.values,
            "status": "no_pending_interrupt",
            "message": "该 thread_id 没有待恢复的中断（请先调用 /start/{thread_id}）",
        }

    result = graph.invoke(
        None,
        config=config,
    )

    return {
        "state": result,
        "status": "completed",
    }


@app.get("/health")
def health():
    """健康检查：自测脚本用它确认服务真的起来了（比 sleep 猜时间可靠）。"""
    return {"ok": True, "port": PORT}

服务在**子线程**里跑（`uvicorn.Server.run()` 只在主线程装信号处理器，塞进线程才不会报
「signal only works in main thread」），主线程用 `/health` 轮询就绪，再打接口，最后 `should_exit` 收工。

In [ ]:
# ---------- 6.3 起服务（子线程）+ 轮询就绪 + 自测 + 收工 ----------
def _start_server_in_thread(port: int):
    """在子线程里启动 uvicorn，返回 (server, thread)。

    uvicorn.Server.run() 只在主线程安装信号处理器
    （框架内部判断 `threading.current_thread() is threading.main_thread()`），
    所以塞进子线程不会报「signal only works in main thread」。
    """
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, name="uvicorn-selftest", daemon=True)
    thread.start()
    return server, thread


def _wait_until_ready(port: int, timeout: float = 20.0) -> bool:
    """轮询 /health，直到服务就绪（或用完超时）。"""
    import requests

    # 为什么不用 time.sleep(2) 死等？uvicorn 在子线程里的启动耗时不可控
    # （首次导入 FastAPI 路由、端口绑定、事件循环起来都要时间），
    # 猜短了会连不上，猜长了每次自测都白等。轮询 /health 是确定性的做法。
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            # /health 是专门为自测加的最轻接口：不碰图、不碰状态，只证明进程活着
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=1).status_code == 200:
                return True
        except requests.RequestException:
            # 服务还没起来时连接会被拒绝（ConnectionError/ReadTimeout 都是它的子类），
            # 这正是预期路径：吞掉异常 → 睡 200ms → 再试，而不是让异常冒泡打断自测
            time.sleep(0.2)
    return False


def self_test() -> None:
    import requests

    print("=" * 74)
    print(f"① 在子线程里启动 uvicorn：http://127.0.0.1:{PORT}")
    print("=" * 74)
    server, thread = _start_server_in_thread(PORT)

    if not _wait_until_ready(PORT):
        print(f"  服务在 20 秒内没起来，可能端口 {PORT} 被占用。")
        print("  排查：换个端口改本文件顶部的 PORT，或用 netstat -ano | findstr 8021 看占用。")
        server.should_exit = True
        thread.join(timeout=5)
        return

    print("  服务已就绪（/health 返回 200）。")
    print()

    base = f"http://127.0.0.1:{PORT}"
    thread_id = "demo-1"

    try:
        # ---------- 3.1 /start：图会停在断点 ----------
        print("=" * 74)
        print(f"② POST /start/{thread_id}  —— 发起任务，预期停在断点")
        print("=" * 74)
        resp = requests.post(f"{base}/start/{thread_id}", timeout=15)
        print(f"  HTTP {resp.status_code}")
        print(f"  返回：{resp.json()}")
        print("  ↑ status=waiting、next=['step']：图停住了，等人工审核。")
        print("    注意 text 仍是 'hello'，step 节点没跑。")
        print()

        # 断点期间再查一次状态：模拟前端「审核页面」刷新
        print("  此时再 POST 一次 /start/demo-2（另一个会话）来对比隔离性：")
        resp2 = requests.post(f"{base}/start/demo-2", timeout=15)
        print(f"  返回：{resp2.json()}")
        print("  ↑ 两个 thread_id 各自一份中断现场，互不干扰。")
        print()

        # ---------- 3.2 /resume：从断点继续 ----------
        print("=" * 74)
        print(f"③ POST /resume/{thread_id}  —— 人工审核通过，恢复执行")
        print("=" * 74)
        resp = requests.post(f"{base}/resume/{thread_id}", timeout=15)
        print(f"  HTTP {resp.status_code}")
        print(f"  返回：{resp.json()}")
        print("  ↑ status=completed、text='hello → 已执行'：断点已放行，step 跑完了。")
        print()

        # ---------- 3.3 重复 resume：校验生效 ----------
        print("=" * 74)
        print("④ 再 POST 一次 /resume  —— 校验「没有待恢复中断」的返回")
        print("=" * 74)
        resp = requests.post(f"{base}/resume/{thread_id}", timeout=15)
        print(f"  返回：{resp.json()}")
        print("  ↑ no_pending_interrupt：接口对无效恢复请求给出了明确提示，而不是静默跑飞。")
        print()
    finally:
        # ---------- 3.4 收工：让 uvicorn 退出 ----------
        print("=" * 74)
        print("⑤ 关闭服务：server.should_exit = True")
        print("=" * 74)
        server.should_exit = True
        thread.join(timeout=10)
        print(f"  uvicorn 已退出（线程存活：{thread.is_alive()}）")
        print()

    print("=" * 74)
    print("小结：脚本版靠 input() 等人按回车，接口版靠 HTTP 请求等人点按钮；")
    print("      两者背后都是同一个 checkpointer 保存的「中断现场」。")
    print("      真实项目里把 MemorySaver 换成 PostgresSaver，恢复逻辑完全不用改。")
    print("=" * 74)

In [ ]:
self_test()

### 预期输出

```text
① 在子线程里启动 uvicorn：http://127.0.0.1:8021
  服务已就绪（/health 返回 200）。

② POST /start/demo-1  —— 发起任务，预期停在断点
  HTTP 200
  返回：{'state': {'text': 'hello'}, 'next': ['step'], 'status': 'waiting'}
  此时再 POST 一次 /start/demo-2（另一个会话）来对比隔离性：
  返回：{'state': {'text': 'hello'}, 'next': ['step'], 'status': 'waiting'}

③ POST /resume/demo-1  —— 人工审核通过，恢复执行
     [服务端] step节点开始执行
  HTTP 200
  返回：{'state': {'text': 'hello → 已执行'}, 'status': 'completed'}

④ 再 POST 一次 /resume  —— 校验「没有待恢复中断」的返回
  返回：{'state': {'text': 'hello → 已执行'}, 'status': 'no_pending_interrupt', 'message': '该 thread_id 没有待恢复的中断（请先调用 /start/{thread_id}）'}

⑤ 关闭服务：server.should_exit = True
  uvicorn 已退出（线程存活：False）
```

注意 `next` 在 Python 里是元组 `('step',)`，过 JSON 序列化后变成数组 `['step']` ——
前端拿到的永远是数组，别按元组去比。

## 7. 官方文档补充：记忆管理、持久化粒度与中断进阶（全离线）

这一节对照 LangGraph 官方文档补课案的空白，四个 Demo **全部离线、0 次真实模型调用**：

| Demo | 官方出处 | 补什么 |
|---|---|---|
| 1 | add-memory.mdx | 记忆「怎么不爆」：裁剪 `trim_messages` / 删除 `RemoveMessage` / 摘要 |
| 2 | checkpointers.mdx | Durability 三档：检查点写几次由你定 |
| 3 | interrupts.mdx | 多中断并行：用「中断 id → 决定」字典一次恢复多个 |
| 4 | interrupts.mdx | 工具内中断 + 中断的三条规则 |

课案讲了记忆怎么**存**，这里讲记忆怎么**不爆**；课案讲了单点中断，这里讲并行中断和工具内中断。

In [ ]:
# ---------- 7.1 导入（Demo 1~4 共用） ----------
from langchain.agents import create_agent
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
    trim_messages,
)
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt
from pydantic import PrivateAttr

### 7.1 Demo 1：记忆「怎么不爆」—— 裁剪 / 删除 / 摘要

对话一长就撑爆上下文。三种手法：
- **裁剪 `trim_messages`**：按 token 数砍掉旧消息，本次请求变短，state 不动；
- **删除 `RemoveMessage`**：按 id 精确删掉 state 里的消息（reducer 只支持增，删只能靠它）；
- **摘要 summarize 节点**：把早期对话压成一段文字存进自定义 state 字段，原文删掉。

In [ ]:
# ---------- 7.2 Demo 1：记忆不爆的三种手法 ----------
class ChatState(MessagesState):
    """MessagesState 自带 messages 字段（用 add_messages reducer）；
    这里再挂一个 summary 字段，用来存压缩后的早期对话。"""

    summary: str


def trim_before_model(state: ChatState) -> dict:
    """节点里演示裁剪：只看不改 —— 计算「如果只保留最近 N 条会是什么样」。"""
    trimmed = trim_messages(
        state["messages"],
        strategy="last",          # 从最新往回保留
        token_counter=len,        # 教学用「条数」当 token 计数（真实项目用模型或估算函数）
        max_tokens=3,             # 预算 3 格 —— SystemMessage 也占格，实际留不到 3 条
        start_on="human",         # 保证第一条**非 system** 消息是 human（否则模型看到半截对话）
        include_system=True,      # 系统提示永远保留
        allow_partial=False,      # 不切半条消息
    )
    print(f"    [trim] 原始 {len(state['messages'])} 条 → 裁到 {len(trimmed)} 条")
    for message in trimmed:
        print(f"        {message.type:<7} {str(message.content)[:28]}")
    return {}   # 只演示，不改状态


def summarize_and_delete(state: ChatState) -> dict:
    """官方 summarize 模式：把早期对话压成摘要写进 state，再把原文删掉。

    注意：本文件为了**离线可复现**，摘要用的是拼好的固定中文（不调模型）；
    生产里这一步就是一次模型调用（官方 SummarizationMiddleware 会自动做）。
    """
    old_messages = state["messages"][:2]          # 假设最早的 2 条要被压缩掉
    summary = state.get("summary") or ""
    new_summary = (summary + " / " if summary else "") + "用户之前问过背景问题并得到了答复"
    # RemoveMessage(id=...) 是唯一能从 state 里**删**消息的手段（reducer 只支持增，不支持删）
    removals = [RemoveMessage(id=m.id) for m in old_messages]
    print(f"    [summarize] 删除 {len(removals)} 条旧消息，摘要字段更新为：{new_summary}")
    return {"messages": removals, "summary": new_summary}


memory_graph = (
    StateGraph(ChatState)
    .add_node("trim", trim_before_model)
    .add_node("summarize", summarize_and_delete)
    .add_edge(START, "trim")
    .add_edge("trim", "summarize")
    .add_edge("summarize", END)
    .compile(checkpointer=InMemorySaver())
)

### 7.2 Demo 2：Durability modes —— 检查点写几次由你定

`durability` 参数有三档：`"sync"`（每步同步落盘，最稳最慢）、`"async"`（异步落盘）、
`"exit"`（只在运行退出时落一次盘，最快但没了中间检查点）。

In [ ]:
# ---------- 7.3 Demo 2：Durability 三档 ----------
class StepState(MessagesState):
    step: int


def make_step_node() -> object:
    def step_node(state: StepState) -> dict:
        return {"step": state.get("step", 0) + 1}

    return step_node

### 7.3 Demo 3：多中断并行 —— 一次恢复多个

并行分支各自中断时，一次 invoke 会返回**多个** Interrupt，恢复要用「中断 id → 决定」的字典。

In [ ]:
# ---------- 7.4 Demo 3：多中断并行 ----------
class ParallelState(MessagesState):
    a: str
    b: str


def branch_a(state: ParallelState) -> dict:
    decision = interrupt({"branch": "A", "ask": "A 分支要放行吗？"})
    return {"a": f"A：{decision}"}


def branch_b(state: ParallelState) -> dict:
    decision = interrupt({"branch": "B", "ask": "B 分支要放行吗？"})
    return {"b": f"B：{decision}"}


parallel_graph = (
    StateGraph(ParallelState)
    .add_node("a", branch_a)
    .add_node("b", branch_b)
    .add_edge(START, "a")
    .add_edge(START, "b")      # 两条并行分支，各自都会中断
    .add_edge("a", END)
    .add_edge("b", END)
    .compile(checkpointer=InMemorySaver())
)

### 7.4 Demo 4：工具内中断 + 中断的三条规则

官方明确支持「在工具内部 interrupt」：工具执行到一半需要人给输入（预算、方案、收货地址…），
整个运行暂停，resume 的值**作为 interrupt() 的返回值**回到工具里继续跑。
这里用一个「剧本假模型」（不调真实模型，离线可复现）驱动工具调用。

In [ ]:
# ---------- 7.5 Demo 4：工具内中断 ----------
class ScriptedModel(ChatOpenAI):
    """按剧本依次吐消息的假模型（同 02_langchain/11_内置中间件_官方补充.py 的手法）。

    这里是工具内中断的演示 —— 需要「模型先要求调工具」这一步，用剧本模型最稳。
    """

    _script: list = PrivateAttr(default_factory=list)
    _cursor: int = PrivateAttr(default=0)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        message = self._script[self._cursor]
        self._cursor += 1
        return ChatResult(generations=[ChatGeneration(message=message)])


@tool
def request_budget(purpose: str) -> str:
    """申请一笔预算（金额需要人工确认）。"""
    # 工具执行到一半停下问人：resume 传进来的值就是 interrupt 的返回值
    amount = interrupt({"question": f"「{purpose}」需要多少预算？"})
    return f"「{purpose}」已批准预算 {amount} 元"


def build_budget_agent():
    model = ScriptedModel(model="scripted", api_key="offline", base_url="http://localhost:9")
    model._script = [
        AIMessage(
            content="",
            tool_calls=[{
                "name": "request_budget",
                "args": {"purpose": "团建"},
                "id": "c1",
                "type": "tool_call",
            }],
        ),
        AIMessage(content="预算已办妥。"),
    ]
    return create_agent(model=model, tools=[request_budget], checkpointer=InMemorySaver())

四个 Demo 依次跑（课案原文写在 `if __name__ == "__main__"` 里，这里直接顶格）。

In [ ]:
# ---------- 7.6 依次跑四个 Demo ----------
# ---------- Demo 1 ----------
print("=" * 70)
print("Demo 1：记忆不爆的三种手法 —— 裁剪 / 删除 / 摘要")
print("=" * 70)
history = [
    SystemMessage(content="你是记账助手", id="s1"),
    HumanMessage(content="我昨天花了 30 元吃午饭", id="m1"),
    AIMessage(content="记下了：午饭 30 元", id="m2"),
    HumanMessage(content="今天午饭花了 45 元", id="m3"),
    AIMessage(content="记下了：午饭 45 元", id="m4"),
    HumanMessage(content="这周一共花了多少？", id="m5"),
]
config = {"configurable": {"thread_id": "memory-demo"}}
result = memory_graph.invoke({"messages": history, "summary": ""}, config)
print(f"  处理后的消息：{[str(m.content)[:16] for m in result['messages']]}")
print(f"  summary 字段：{result['summary']}")
print(
    "  ↑ 裁剪（trim）只影响「本次发给模型的内容」，不动 state；\n"
    "    删除（RemoveMessage）+ 摘要才真正压缩 state —— 课案讲过的 SummarizationMiddleware\n"
    "    与 ContextEditingMiddleware 就是这两招的封装版（生产直接用封装，懂原理才排得动）。"
)

# ---------- Demo 2 ----------
print("\n" + "=" * 70)
print("Demo 2：Durability modes —— 检查点写几次由你定")
print("=" * 70)
for mode in ("sync", "async", "exit"):
    graph = (
        StateGraph(StepState)
        .add_node("one", make_step_node())
        .add_node("two", make_step_node())
        .add_node("three", make_step_node())
        .add_edge(START, "one")
        .add_edge("one", "two")
        .add_edge("two", "three")
        .add_edge("three", END)
        .compile(checkpointer=InMemorySaver())
    )
    run_config = {"configurable": {"thread_id": f"durability-{mode}"}}
    out = graph.invoke({"messages": [], "step": 0}, run_config, durability=mode)
    checkpoints = len(list(graph.get_state_history(run_config)))
    print(f"  durability={mode:<6} 最终 step={out['step']}，落盘检查点数量={checkpoints}")
print(
    "  ↑ 实测：sync / async 都写了 5 个检查点，而 exit **只写 1 个**（运行结束才落盘）。\n"
    "    选型：要中断恢复/时间旅行 → sync 或 async（这俩数量相同，差别在写入时机：\n"
    "    sync 写完才继续、async 异步写）；纯批处理、崩了重跑即可 → exit 最快。"
)

# ---------- Demo 3 ----------
print("\n" + "=" * 70)
print("Demo 3：多中断并行 —— 用「中断 id → 决定」的字典一次恢复")
print("=" * 70)
multi_config = {"configurable": {"thread_id": "parallel-interrupt"}}
first = parallel_graph.invoke({"messages": [], "a": "", "b": ""}, multi_config)
interrupts = first.get("__interrupt__", [])
print(f"  第一次 invoke 返回 {len(interrupts)} 个中断：")
for item in interrupts:
    print(f"    id={item.id[:12]}… value={item.value}")
decisions = {item.id: f"{item.value['branch']} 分支已放行" for item in interrupts}
resumed = parallel_graph.invoke(Command(resume=decisions), multi_config)
print(f"  字典恢复后：a={resumed['a']!r} b={resumed['b']!r}")
print(
    "  ↑ 单个中断时 Command(resume=值) 就够；**并行多中断**必须用\n"
    "    Command(resume={中断id: 值}) 把每个决定送回对应的分支。"
)

# ---------- Demo 4 ----------
print("\n" + "=" * 70)
print("Demo 4：工具内中断 —— 工具执行到一半问人要输入")
print("=" * 70)
agent = build_budget_agent()
agent_config = {"configurable": {"thread_id": "tool-interrupt"}}
first = agent.invoke({"messages": [{"role": "user", "content": "帮我办个团建"}]}, agent_config)
print(f"  第一次 invoke 返回中断：{[i.value for i in first.get('__interrupt__', [])]}")
second = agent.invoke(Command(resume=5000), agent_config)
tool_messages = [m for m in second["messages"] if m.type == "tool"]
print(f"  恢复后工具返回：{[str(m.content) for m in tool_messages]}")
print(f"  最终回复：{second['messages'][-1].content}")
print(
    "  ↑ resume 的值（5000）**成了工具里 interrupt() 的返回值**，工具继续跑完。\n"
    "    规则：① 别用 try/except 包 interrupt（暂停靠抛异常实现）；\n"
    "          ② 中断点之前的副作用必须幂等（恢复会重放）；\n"
    "          ③ 同一节点内多次 interrupt 的顺序/次数要稳定。"
)

print("\n全部 Demo 执行完毕（0 次模型调用，离线可复现）。")

### 预期输出

```text
Demo 1：记忆不爆的三种手法 —— 裁剪 / 删除 / 摘要
    [trim] 原始 6 条 → 裁到 2 条
        system  你是记账助手
        human   这周一共花了多少？
    [summarize] 删除 2 条旧消息，摘要字段更新为：用户之前问过背景问题并得到了答复
  处理后的消息：['记下了：午饭 30 元', '今天午饭花了 45 元', '记下了：午饭 45 元', '这周一共花了多少？']
  summary 字段：用户之前问过背景问题并得到了答复

Demo 2：Durability modes —— 检查点写几次由你定
  durability=sync   最终 step=3，落盘检查点数量=5
  durability=async  最终 step=3，落盘检查点数量=5
  durability=exit   最终 step=3，落盘检查点数量=1

Demo 3：多中断并行 —— 用「中断 id → 决定」的字典一次恢复
  第一次 invoke 返回 2 个中断：
  字典恢复后：a='A：A 分支已放行' b='B：B 分支已放行'

Demo 4：工具内中断 —— 工具执行到一半问人要输入
  第一次 invoke 返回中断：[{'question': '「团建」需要多少预算？'}]
  恢复后工具返回：['「团建」已批准预算 5000 元']
  最终回复：预算已办妥。

全部 Demo 执行完毕（0 次模型调用，离线可复现）。
```

⚠️ Demo 3 里那两行 `id=… value={...}` 里 `id` 是 `Interrupt.id[:12]`，**每次运行随机生成**，
别逐字比对；`value`（分支名 + 提问）与 `字典恢复后` 那行是稳定的。其余全部离线可复现、逐字一致。

下面这格是本 notebook 的**最后一个 code cell**：关掉第 5 节后台拉起的 8110 服务。
Windows 上要连子进程树一起收（`taskkill /F /T /PID`），否则 uvicorn 的 worker 会成为孤儿进程占着端口。

In [ ]:
# ===== 收尾：关掉 8110 的后台服务（连子进程树一起收）=====
if _api_server is not None and _api_server.poll() is None:
    subprocess.run(["taskkill", "/F", "/T", "/PID", str(_api_server.pid)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    _api_server.wait(timeout=10)
    print("8110 服务已关闭（含子进程树）。")
else:
    print("8110 服务未在运行（无需关闭）。")

## 小结

- **流式**靠 `graph.stream(stream_mode=...)`：`messages` 给 token（体验最好）、
  `updates` 给增量（最省）、`values` 给全量（最全）；多模式同开时事件是 `(模式, 数据)` 二元组；
- **动态中断** `interrupt()` 在节点内暂停、把问题抛给人类，用 `Command(resume=值)` 恢复；
- **静态断点** `interrupt_before` 在编译时写死关卡，用 `invoke(None, config)` 放行；
- **两者都必须配 checkpointer**：能恢复不是断点的功劳，是 checkpointer 保存「中断现场」的功劳；
- **接口版**把「按回车」换成「点按钮」：中断现场跨 HTTP 请求保存在服务端，`thread_id` 是取件码；
- **官方补充**：记忆裁剪/删除/摘要控制上下文不爆、Durability 三档控制检查点写几次、
  并行多中断用 `{中断id: 值}` 字典恢复、工具内也能 `interrupt()`。

下一课 `04_时间旅行_子图与容错.ipynb` 会看到：有了 checkpointer，还能**回到历史检查点**重放图。

## 常见坑

1. **`messages` 模式是拦截回调事件实现的**：节点里 `llm.invoke()`（非流式）也能逐 token 吐，
   不用为了流式改业务代码；但 `message_chunk.content` 不一定是字符串（可能分块列表），要先判类型。
2. **静态断点恢复用 `invoke(None, config)`，不是 `Command(resume=...)`**；动态中断才用 `Command`。
   两者混用「看起来跑通其实没恢复」，且不报错，极难排查。
3. **恢复必须用同一个 `thread_id`**：换一个 thread_id 就找不到现场，`invoke(None)` 拿空状态往下跑，
   行为与预期完全不同且不报错。
4. **常驻服务（`uvicorn.run`）会卡死内核**：必须后台起（`subprocess.Popen`）+ 末尾 `taskkill /F /T`，
   否则 uvicorn 的 worker 会成为孤儿进程占着端口。
5. **`input()` 在无头执行必崩**：非交互环境 `stdin.isatty()` 为 False，直接 `input()` 会 EOFError；
   要么像第 4 节 `wait_for_enter()` 那样加 isatty + EOFError 兜底，要么用预设答案函数替换 `builtins.input`。
6. **`durability="exit"` 牺牲中间检查点**：时间旅行与崩溃后中途续跑都没了，但中断恢复仍可用
   （退出那一刻会落盘）。
7. **`interrupt()` 不能被 try/except 包住**：暂停靠抛异常实现，被吞掉后 resume 再也走不动。
8. **Windows 上刚 taskkill 的进程句柄不立刻释放**：删临时目录会 `WinError 32`，要重试 + 容忍失败。

## 官方链接

- 流式输出（streaming）：<https://docs.langchain.com/oss/python/langgraph/streaming>
- 中断 / 人工审核（interrupts）：<https://docs.langchain.com/oss/python/langgraph/interrupts>
- 持久化 / 检查点（persistence）：<https://docs.langchain.com/oss/python/langgraph/persistence>
- 记忆管理（Manage short-term memory）：<https://docs.langchain.com/oss/python/langgraph/add-memory>
- Durability modes（checkpointers）：<https://docs.langchain.com/oss/python/langgraph/checkpointers>